# 💰 Expense Tracker App – Exploratory Data Analysis
**Project:** Expense Tracker using Data Science  
**Author:** [Your Name]  
**Goal:** Understand spending patterns, detect overspending, and generate actionable insights.

---

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

print('✅ Libraries loaded')

## 2. Generate & Load Data

In [ ]:
from data_generator import generate_expense_data, introduce_quality_issues
from data_cleaning  import clean_pipeline

os.makedirs('../data', exist_ok=True)

# Generate
df_clean = generate_expense_data(num_records=500)
df_dirty = introduce_quality_issues(df_clean)
df_clean.to_csv('../data/expenses_clean.csv',  index=False)
df_dirty.to_csv('../data/expenses_raw.csv',    index=False)

# Clean
df = clean_pipeline('../data/expenses_raw.csv', '../data/expenses_cleaned.csv')
df.head()

## 3. Dataset Overview

In [ ]:
print('Shape:', df.shape)
print('\nData Types:')
print(df.dtypes)
print('\nNull Values:')
print(df.isnull().sum())

In [ ]:
df.describe().round(2)

## 4. Category-wise Spending

In [ ]:
cat = df.groupby('Category')['Amount'].sum().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Bar chart
axes[0].barh(cat.index, cat.values, color=sns.color_palette('Set2', len(cat)))
axes[0].set_title('Category Total Spending', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Amount (₹)')

# Pie chart
axes[1].pie(cat.values, labels=cat.index, autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('Set2', len(cat)))
axes[1].set_title('Spending Share by Category', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Monthly Trend Analysis

In [ ]:
MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

monthly = df.groupby('Month')['Amount'].sum().reindex(MONTH_ORDER).dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar
axes[0].bar(monthly.index, monthly.values, color=sns.color_palette('Blues_d', 12))
axes[0].axhline(30000, color='red', linestyle='--', label='Budget ₹30,000')
axes[0].set_title('Monthly Total Expenses', fontsize=14, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# Line
axes[1].plot(monthly.index, monthly.values, marker='o', linewidth=2.5, color='#2196F3')
axes[1].fill_between(monthly.index, monthly.values, alpha=0.15, color='#2196F3')
axes[1].axhline(30000, color='red', linestyle='--', label='Budget ₹30,000')
axes[1].set_title('Monthly Expense Trend', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Heatmap – Category × Month

In [ ]:
pivot = df.pivot_table(index='Category', columns='Month_Num', values='Amount', aggfunc='sum').fillna(0)
pivot.columns = [f'M{c}' for c in pivot.columns]

plt.figure(figsize=(16, 7))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5)
plt.title('Spending Heatmap: Category × Month', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Payment Method & Day-of-Week Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Payment method
pay = df.groupby('Payment_Method')['Amount'].sum()
axes[0].pie(pay.values, labels=pay.index, autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('Pastel1', len(pay)), wedgeprops={'width': 0.5})
axes[0].set_title('Payment Method Distribution', fontsize=14, fontweight='bold')

# Day of week
DOW = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day = df.groupby('Day_of_Week')['Amount'].sum().reindex(DOW)
colors = ['#FF7043' if d in ['Saturday','Sunday'] else '#42A5F5' for d in DOW]
axes[1].bar(day.index, day.values, color=colors)
axes[1].set_title('Spending by Day of Week', fontsize=14, fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 8. Budget vs Actual & Cumulative Spend

In [ ]:
BUDGET = 30_000
mdf = df.groupby(['Month_Num','Month'])['Amount'].sum().reset_index().sort_values('Month_Num')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Budget vs Actual
x  = range(len(mdf))
axes[0].bar([i - 0.2 for i in x], [BUDGET]*len(mdf), 0.4, label='Budget', color='#66BB6A', alpha=0.85)
axes[0].bar([i + 0.2 for i in x], mdf['Amount'], 0.4, label='Actual',
            color=['#EF5350' if a > BUDGET else '#42A5F5' for a in mdf['Amount']])
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(mdf['Month'], rotation=45)
axes[0].set_title('Budget vs Actual', fontsize=14, fontweight='bold')
axes[0].legend()

# Cumulative
cumulative = df.groupby('Date')['Amount'].sum().sort_index().cumsum()
axes[1].plot(cumulative.index, cumulative.values, color='#AB47BC', linewidth=2)
axes[1].fill_between(cumulative.index, cumulative.values, alpha=0.15, color='#AB47BC')
axes[1].set_title('Cumulative Expense', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Key Insights Summary

In [ ]:
from analysis import generate_insights
insights = generate_insights(df)
print('\n'.join(insights))

---
## ✅ Notebook Complete!
- All data generated, cleaned, analysed, and visualised.
- Run `python main.py` from the project root to regenerate everything.
